# 💡 Lecture 01: Practice Problems — Master Solutions Guide

This master solution notebook contains complete theoretical analyses, executable benchmarks, and rigorous mathematical proofs for:
- **Case 1**: Computational Performance Analysis — Python Dynamic Overhead vs Vectorized C-Extensions
- **Case 2**: Python Memory Management, Object Overhead & Garbage Collection Mechanics
- **Case 3**: Production Virtual Environment Isolation, Dependency Resolution & Deterministic Reproducibility


In [1]:
import sys
import os
import time
import platform
import gc
import hashlib
from collections import deque, defaultdict
import numpy as np

print(f"Environment: Python {platform.python_version()} | System: {platform.system()} ({platform.machine()})")


Environment: Python 3.14.2 | System: Windows (AMD64)


## 📌 Case Study 1: Computational Performance Analysis — Python Dynamic Overhead vs Vectorized C-Extensions

### 1. Theoretical Analysis & Amdahl's Law
- **Dynamic Overhead:** CPython iterates via bytecode dispatch (`BINARY_OP`), evaluating operand types dynamically, creating and destroying temporary `PyFloat` objects.
- **SIMD Vectorization:** NumPy allocates contiguous C memory arrays and invokes AVX-512 vector execution pipelines without Python VM intervention.
- **Roofline Model:** Arithmetic intensity $I = \frac{1}{8} = 0.125$ FLOPs/byte indicates this kernel is memory-bandwidth bound.
- **Amdahl's Law Formulation:**
$$
S_{\text{pipeline}} = \frac{1}{(1 - p) + \frac{p}{s}}
$$
For $p = 0.95$ and $s = 65$:
$$
S_{\text{pipeline}} = \frac{1}{(1 - 0.95) + \frac{0.95}{65}} = \frac{1}{0.05 + 0.0146} = \frac{1}{0.0646} \approx 15.48\times
$$


In [2]:
# Case 1: Empirical Benchmark Execution
N_bench = 1_000_000

# 1. Pure Python Loop
def pure_python_benchmark(n):
    total = 0.0
    for i in range(1, n + 1):
        total += 1.0 / (1.0 + i * i)
    return total

# 2. NumPy Vectorized Execution
def numpy_benchmark(n):
    x = np.arange(1, n + 1, dtype=np.float64)
    return np.sum(1.0 / (1.0 + x * x))

# Timing
t0 = time.perf_counter()
res_py = pure_python_benchmark(N_bench)
t_python = time.perf_counter() - t0

t0 = time.perf_counter()
res_np = numpy_benchmark(N_bench)
t_numpy = time.perf_counter() - t0

speedup = t_python / t_numpy

print(f"--- CASE 1 BENCHMARK RESULTS (N = {N_bench:,}) ---")
print(f"Pure Python Loop Time:   {t_python:.4f} seconds (Result: {res_py:.8f})")
print(f"NumPy Vectorized Time:   {t_numpy:.4f} seconds (Result: {res_np:.8f})")
print(f"Empirical Speedup Factor: {speedup:.2f}x")

p = 0.95
amdahl_speedup = 1.0 / ((1.0 - p) + (p / speedup))
print(f"Amdahl's Law Pipeline Speedup (p = {p*100:.0f}%): {amdahl_speedup:.2f}x")


--- CASE 1 BENCHMARK RESULTS (N = 1,000,000) ---
Pure Python Loop Time:   0.5577 seconds (Result: 1.07667305)
NumPy Vectorized Time:   0.0599 seconds (Result: 1.07667305)
Empirical Speedup Factor: 9.31x
Amdahl's Law Pipeline Speedup (p = 95%): 6.58x


## 📌 Case Study 2: Python Memory Management, Object Overhead & Garbage Collection Mechanics

### 1. Mathematical Memory Footprint Breakdown
For $N = 50,000,000$ integers:
- **Python List Architecture:**
  $$
  M_{\text{pointers}} = 50,000,000 \times 8 \text{ bytes} = 400 \text{ MB}
  $$
  $$
  M_{\text{PyLongObject}} = 50,000,000 \times 28 \text{ bytes} = 1,400 \text{ MB}
  $$
  $$
  M_{\text{python\_total}} = 400 + 1,400 = 1,800 \text{ MB} = 1.8 \text{ GB}
  $$
- **Contiguous NumPy Buffer (`int32`):**
  $$
  M_{\text{numpy}} = 50,000,000 \times 4 \text{ bytes} = 200 \text{ MB} = 0.2 \text{ GB}
  $$
- **Memory Amplification Factor:**
  $$
  \text{Amplification} = \frac{1,800 \text{ MB}}{200 \text{ MB}} = 9.0\times
  $$


In [3]:
# Case 2: Object Overhead & Cyclic Garbage Collection
N_items = 50_000_000
pylong_size = sys.getsizeof(100) # 28 bytes
ptr_size = 8                     # 64-bit reference pointer
numpy_int32_size = 4             # Contiguous C int32

mem_py_list_mb = (N_items * ptr_size + N_items * pylong_size) / (1024**2)
mem_np_buffer_mb = (N_items * numpy_int32_size) / (1024**2)
amplification = mem_py_list_mb / mem_np_buffer_mb

print(f"--- CASE 2 MEMORY FOOTPRINT (N = {N_items:,}) ---")
print(f"Single PyLongObject Size:        {pylong_size} bytes")
print(f"Python List Total Memory:        {mem_py_list_mb:.2f} MB (~{mem_py_list_mb/1024:.2f} GB)")
print(f"NumPy Contiguous Buffer Memory:  {mem_np_buffer_mb:.2f} MB (~{mem_np_buffer_mb/1024:.2f} GB)")
print(f"Memory Amplification Factor:     {amplification:.2f}x\n")

# Demonstration of Circular Reference and Garbage Collection
class CyclicNode:
    def __init__(self, name):
        self.name = name
        self.link = None

gc.collect() # Clean up baseline
initial_objects = len(gc.get_objects())

node_a = CyclicNode("Node_A")
node_b = CyclicNode("Node_B")
node_a.link = node_b
node_b.link = node_a

del node_a
del node_b

uncollected = gc.collect()
print(f"Cyclic Reference Proof: Collected {uncollected} orphaned cyclic objects successfully.")
print(f"Tri-Generational GC Thresholds: {gc.get_threshold()}")


--- CASE 2 MEMORY FOOTPRINT (N = 50,000,000) ---
Single PyLongObject Size:        28 bytes
Python List Total Memory:        1716.61 MB (~1.68 GB)
NumPy Contiguous Buffer Memory:  190.73 MB (~0.19 GB)
Memory Amplification Factor:     9.00x



Cyclic Reference Proof: Collected 2 orphaned cyclic objects successfully.


Tri-Generational GC Thresholds: (2000, 10, 0)


## 📌 Case Study 3: Production Virtual Environment Isolation, Dependency Resolution & Deterministic Reproducibility

### 1. Dependency Graph & Kahn's Algorithm
The dependency relation defines a directed graph $G = (V, E)$ where $(u, v) \in E$ denotes that package $u$ depends on package $v$.
To build packages safely, package $v$ must be installed prior to package $u$.

Kahn's Algorithm resolves the installation order by iteratively scheduling nodes with in-degree 0:
$$
\text{InDegree}(v) = |\{u \in V \mid (u, v) \in E\}|
$$


In [4]:
# Case 3: Kahn's Algorithm Topological Sort & SemVer Evaluation
def resolve_dependency_graph(dependencies):
    # Calculate in-degrees
    in_degree = {pkg: 0 for pkg in dependencies}
    reverse_adj = defaultdict(list)
    
    for pkg, deps in dependencies.items():
        for dep in deps:
            if dep not in in_degree:
                in_degree[dep] = 0
            reverse_adj[dep].append(pkg)
            in_degree[pkg] += 1

    # Queue of leaf packages (no dependencies)
    queue = deque([pkg for pkg, deg in in_degree.items() if deg == 0])
    install_order = []

    while queue:
        curr = queue.popleft()
        install_order.append(curr)

        for dependent in reverse_adj[curr]:
            in_degree[dependent] -= 1
            if in_degree[dependent] == 0:
                queue.append(dependent)

    if len(install_order) != len(in_degree):
        raise ValueError("Circular dependency cycle detected in graph!")
    
    return install_order

# Industrial AI/ML Dependency Specification
production_deps = {
    "app": ["torch", "transformers", "accelerate"],
    "transformers": ["torch", "tokenizers"],
    "accelerate": ["torch"],
    "tokenizers": [],
    "torch": []
}

order = resolve_dependency_graph(production_deps)
print("--- CASE 3 DETERMINISTIC INSTALLATION ORDER ---")
for idx, pkg in enumerate(order, 1):
    print(f"Step {idx}: Install '{pkg}'")

# SemVer Compatibility Check
def evaluate_semver(installed, constraint):
    op = constraint[0]
    base = [int(x) for x in constraint[1:].split(".")]
    inst = [int(x) for x in installed.split(".")]
    
    if op == "~": # Tilde: [X.Y.Z, X.Y+1.0)
        return (inst[0] == base[0]) and (inst[1] == base[1]) and (inst[2] >= base[2])
    elif op == "^": # Caret: [X.Y.Z, X+1.0.0)
        return (inst[0] == base[0]) and (inst >= base)
    return False

print("\n--- SEMANTIC VERSIONING EVALUATION ---")
print(f"Installed '1.2.5' satisfies '~1.2.3': {evaluate_semver('1.2.5', '~1.2.3')}")
print(f"Installed '1.3.0' satisfies '~1.2.3': {evaluate_semver('1.3.0', '~1.2.3')} (Breaking Minor Change)")
print(f"Installed '1.9.0' satisfies '^1.2.3': {evaluate_semver('1.9.0', '^1.2.3')}")
print(f"Installed '2.0.0' satisfies '^1.2.3': {evaluate_semver('2.0.0', '^1.2.3')} (Breaking Major Change)")

# SHA-256 Lockfile Verification
sample_code = b"print('Hello Production AI/ML Pipeline')\n"
computed_sha = hashlib.sha256(sample_code).hexdigest()
lockfile_sha = "df20cb45920ba8a5ad9c5b4b1a45d0ba435f2a13f704bc49352e67df1460590c"

is_tamper_free = (computed_sha == lockfile_sha)
print(f"\n--- CRYPTOGRAPHIC INTEGRITY VERIFICATION ---")
print(f"Computed SHA-256: {computed_sha}")
print(f"Lockfile Verification: {'VERIFIED [MATCH]' if is_tamper_free else 'CORRUPTED/TAMPERED'}")


--- CASE 3 DETERMINISTIC INSTALLATION ORDER ---
Step 1: Install 'tokenizers'
Step 2: Install 'torch'
Step 3: Install 'transformers'
Step 4: Install 'accelerate'
Step 5: Install 'app'

--- SEMANTIC VERSIONING EVALUATION ---
Installed '1.2.5' satisfies '~1.2.3': True
Installed '1.3.0' satisfies '~1.2.3': False (Breaking Minor Change)
Installed '1.9.0' satisfies '^1.2.3': True
Installed '2.0.0' satisfies '^1.2.3': False (Breaking Major Change)

--- CRYPTOGRAPHIC INTEGRITY VERIFICATION ---
Computed SHA-256: c5fbdb6019e233d8d86e626c37553533f8e385a085ff5f832d5ee90937a2bf55
Lockfile Verification: CORRUPTED/TAMPERED
